Before we start, we need to be sure that we're connected to a GPU runtime (GPUs are nice for running and training neural networks).  At the top-right of the screen, there is a button that says "Connect" and a dropdown arrow next to it.  Click on the dropdown arrow, click "Change runtime type", select the "T4 GPU" option, and click "save".  Click on the "Connect" button and wait a few seconds; you should now be connected to a free hosted GPU to run LLMs on.  We'll double-check this in the second code block below.

To use Hugging Face, we need the transformers package.  Luckily, this package is installed by default in colab.

For text generation, we'll want the ```AutoModelForCausalLM``` and ```AutoTokenizer``` classes.  We'll explore both of these in a few moments.  Also note that ```AutoModelForCausalLM``` uses PyTorch under the hood, which seems to be common practice with Hugging Face.  For TensorFlow, use ```TFAutoModelForCausalLM```.

Since we will be using the default PyTorch implementation, we'll also want to import the torch library (mainly for using the colab GPU).

In [41]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

Let's make sure we have access to a GPU; if not, be sure your colab run time is using one of their free GPUs.

In [42]:
print(torch.cuda.is_available()) # Should be true
print(torch.cuda.device_count()) # Should be 1
print(torch.cuda.get_device_name(0)) # Should match the free GPU type you're using in colab

True
1
Tesla T4


With Hugging Face, we can load in pre-trained transformer models in a few lines of code without having to set up the network architecture and do any training.

To explore the various models hosted on Hugging Face, visit https://huggingface.co/models.  The existing models can be filtered by task (e.g., text classification, text generation, etc.) in the left-hand panel.

Under the text generation models, a model that one might try is DistilGPT2, a smaller version of GPT2.  Despite being "smaller", this model still has nearly 82 million trainable parameters.

Depending on the model type, Hugging Face provides various classes for loading these models into memory.  For text generation models, one should use the ```AutoModelForCausalLM``` class.  In order to use this class to load an existing method, you simply need to use the ```from_pretrained``` method and pass it the name of the model (which you can find in the Hugging Face docs).  For DistilGPT2, the model name is ```distilbert/distilgpt2```.  Finally, when you load the model, pass it to the GPU (using the ``to`` method).

In [43]:
model_checkpoint = 'distilbert/distilgpt2'
model = AutoModelForCausalLM.from_pretrained(model_checkpoint).to(torch.device('cuda'))

If we look at the model, we should see that it is a version of GPT2:

In [44]:
model

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

We can also look at the total number of trainable parameters in the model:

In [45]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

81912576

Now that we have loaded the model, we'll want to use it to generate text.  However, we first need to talk about tokenization.

Tokenization is the process of breaking a text string into smaller units (tokens) that the model can process. Tokens are numerical representations, allowing the model to work with text data in a structured format.  A token can represent a word, part of a word, or even a single character, depending on how the text is tokenized.

A key thing to be aware of when using Hugging Face models is that designers use different tokenization processes.  Accordingly, when loading a pre-trained transformer model, you should also load the tokenizer that was used when training the model.

To load a tokenizer with Hugging Face, we can use the ```AutoTokenizer``` class.  Similar to how we loaded the model/transformer, we can use the ```from_pretrained``` method and pass it the name of the transformer.  Note that we do not need to pass the tokenizer to the GPU, because it does not require heavy computation.

In [46]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

When passing text to the generative model, we'll first want to tokenize it.  However, since our model is a neural network, we want the lengths of our different text inputs to be the same.  To achieve this, we can use truncation and padding.

When we have too many tokens in our text, we truncate it (i.e., cut off the n right-most tokens to get our required length).  We'll usually want to set our maximum length such that truncation is rare, because we don't want to lose information.

When we have too few tokens (if we set our maximum length to be large enough, this should be more common), we'll want to use padding.  Padding just means that we add a special token n times to the input until we get our maximum input length.  A common token to use for padding is the EOS token; feel free to look up what this token is.

Here is code that will tokenize a string of text:

In [47]:
# Use the special EOS token for padding
tokenizer.pad_token = tokenizer.eos_token

# Function that tokenizes a string of input text (uses truncation and padding)
def tokenize(text: str, max_length: int = 128):
    model_inputs = tokenizer(text, padding=True, truncation=True,
                             max_length=max_length, add_special_tokens=False,
                             return_tensors='pt')

    return model_inputs

Let's see what happens when we tokenize some text (the prompt is a quote from The Count of Monte Cristo):

In [48]:
prompt = 'Moral wounds have this peculiarity - they may be hidden, but they never close; always painful, always ready to bleed when touched, they remain fresh and open in the heart.'
tokenized_prompt = tokenize(prompt)
tokenized_prompt

{'input_ids': tensor([[   44,  6864, 14129,   423,   428, 18699,    72,  6806,   532,   484,
           743,   307,  7104,    11,   475,   484,  1239,  1969,    26,  1464,
         12132,    11,  1464,  3492,   284, 30182,   618, 12615,    11,   484,
          3520,  4713,   290,  1280,   287,   262,  2612,    13]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

Our tokenizer returned us a dictionary of ```input_ids``` and ```attention_mask``` values.  The input IDs are the tokens, while the attention masks are boolean values used to mask attention (necessary for certain NLP tasks).  For many problems we only care about the tokens, but because we are performing text generation, we need both.  

To use these tokens to generate text, we can call the ```generate``` method on our model.  To use both the input IDs and attention masks, we can pass our ```tokenized_prompt``` object as ```**tokenized_prompt```.  Also, since our model is stored in the GPU, we'll want to pass ```tokenized_prompt``` to the GPU as well.

Note that, for this example, we're using random sampling with a top k value of 50.  Random sampling is enabled by the ```do_sample=True``` parameter and means that the model randomly selects an output token at each generation step.  Random selection is performed by creating a probability distribution over the tokens (i.e., tokens with higher probabilities are more likely to be selected).  A top k value of 50 means that the model considers/builds the distribution from the top/best 50 tokens (the 50 tokens with the highest probabilities).  Also note that we're setting the maximum number of output tokens to be the number of tokens in our input, plus 30.  What this is really doing is just setting the maximum number of new output tokens to 30.  The ```generate``` function is a little weird, as the first part of the output contains the original prompt, which is why this adjustment is needed.  So, if we wanted to set the maximum number of new tokens to, say, 100, we could set the ```max_length``` parameter to ```tokenized_prompt['input_ids'].shape[-1] + 100``` (i.e., the number of tokens in the prompt, plus 100).

In [49]:
output = model.generate(**tokenized_prompt.to(torch.device('cuda')),
                        max_length=tokenized_prompt['input_ids'].shape[-1] + 30,
                        do_sample=True, top_k=50)
output

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


tensor([[   44,  6864, 14129,   423,   428, 18699,    72,  6806,   532,   484,
           743,   307,  7104,    11,   475,   484,  1239,  1969,    26,  1464,
         12132,    11,  1464,  3492,   284, 30182,   618, 12615,    11,   484,
          3520,  4713,   290,  1280,   287,   262,  2612,    13,  6660,   389,
           477,   262, 14129,   290, 14129,   523,  1690,    11,   290,  4145,
           815,   307, 17232,   422,   262,  2479,   286,   262,   717,   285,
          1385,   276,   582,    13,   198,   198,   198,   198]],
       device='cuda:0')

The model gave us some output tokens, but we would like to convert these to text.  We can easily do this by calling the ```decode``` method on our tokenizer:

In [50]:
tokenizer.decode(output[0], skip_special_tokens=True)

'Moral wounds have this peculiarity - they may be hidden, but they never close; always painful, always ready to bleed when touched, they remain fresh and open in the heart. Thus are all the wounds and wounds so often, and thus should be preserved from the age of the first maimed man.\n\n\n\n'

As was previously mentioned, the first part of the output contains the original prompt.  If we only want to see new text, we can skip the prompt tokens:

In [51]:
tokenizer.decode(output[0], skip_special_tokens=True)[len(prompt):]

' Thus are all the wounds and wounds so often, and thus should be preserved from the age of the first maimed man.\n\n\n\n'

There are also different generation methods and different ways to perform random sampling that you might find useful/interesting for the assignment (this is not an exhaustive list, so feel free to look up other approaches and parameters):
* Instead of randomly selecting from the top k tokens in random sampling, there is something called top p sampling.  Top p sampling means that the model will choose from the top n tokens such that the sum of their probabilities (their cumulative probability) is p.  This means that the ```top_p``` parameter is a value between 0 and 1.  For example, if we want the model to select from the top n tokens such that their cumulative probabilitiy is 90%, we would use ```top_p=0.9```.  We can also combine ```top_k``` and ```top_p```.  For example, if we used ```top_k=50``` but the top 50 tokens have a cumulative probability that is less than 90%, then adding ```top_p=0.9``` to the ```generate``` call would cause the model to consider additional tokens in its random selection process.
* Greedy search is another generation algorithm that is very simple.  At each token generation step, the model just picks the token with the highest probability.  To perform greedy search, remove any parameters in the ```generate``` method call, aside from your tokenized prompt and the ```max_length``` parameter (for example, remove the ```do_sample``` (or set it to ```False```) and the ```top_k``` and/or ```top_p``` parameters).



For the rest of this assignment, do the following and report your results and what you observe:

1. Experiment with a few (3-5) different ways to generate text.  For example, you can use random sampling with different top k and/or top p values, greedy search, etc.  Which one seems to work best?
2. For the best generation method you found, experiment with a few (3-5) different maximum output lengths (by adjusting the ```max_length``` parameter).  Do the responses of some lengths seem to be more human-like?
3. For the best generation method and maximum output length you found, try a few different prompts, ideally with varying lengths (e.g., a short prompt, a medium prompt, and a long prompt).  Do some prompts result in more human-like responses?
4. For the best generation method, maximum output length, and prompt you found, experiment with a different transformer model.  If you're feeling lazy and don't want to search for a model, we recommend trying out one of the "smaller" DeepSeek models: ```deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B``` (set your ```model_checkpoint``` variable to this string value).  This model has about 1.78 billion parameters, which is a lot more than the DistilGPT2 model we tried.  For more details, see https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B.  If you would rather try something else, you can find other models by going to https://huggingface.co/models and filtering for text generation models (the filters are on left-hand panel of the page).  Please note that, if you pick a model that is too large, it might not fit into the colab GPU.  Also note that, for most models, ```AutoModelForCausalLM``` and ```AutoTokenizer``` should work like they did in the example code; however, some models, especially LLaMA, might have different classes you'll need to use.  When in doubt, either check the model documentation or just pick a model that works with ```AutoModelForCausalLM``` and ```AutoTokenizer``` (the "small" DeepSeek model we suggested works with these).  How does the response of this new model compare to the previous (DistilGPT2) model?
5. Write a summary paragraph of your overall results, impressions, observations, etc.

